In [ ]:
!pip install -U bitsandbytes>=0.46.1

In [ ]:
!pip install duckdb

# FHIR-SQL Supervised Fine-Tuning -- DoRA on Qwen2.5-Coder-14B-Instruct

In [ ]:
# ==== environment & paths ====
import os, shutil

# Reduces CUDA-allocator fragmentation from variable-length batches (micro_batch_size=1 +
# dynamic padding means every training step's tensor shape differs) -- must be set before
# torch initializes its CUDA context, so this has to run before `import torch` anywhere below.
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

try:
    from google.colab import drive
    drive.mount('/content/drive',force_remount=True)
    ON_COLAB = True
except Exception:
    ON_COLAB = False

# RunPod: RUNPOD_POD_ID is set in every RunPod container. Uses a persistent network
# volume (survives pod stop/start) mounted at /workspace, so -- unlike Colab's Drive FUSE mount,
# which has real sync lag -- writes to ROOT here are plain local-disk writes, durable without
# needing to round-trip every checkpoint back to Drive. rclone is only used once, to PULL the
# read-only inputs (model weights, schema, DBs, training data) onto the volume the first time
# it's used.
ON_RUNPOD = 'RUNPOD_POD_ID' in os.environ

if ON_COLAB:
    _default_root = '/content/drive/MyDrive/projects/FHIRSQL'
elif ON_RUNPOD:
    _default_root = '/workspace/FHIRSQL'
else:
    _default_root = 'C:/dev/fhirsql-phase2'
ROOT = os.environ.get('FHIRSQL_ROOT', _default_root)
LOCAL_DIR = '/content/local' if ON_COLAB else ROOT   # Colab needs a real local copy (FUSE read latency);
                                                       # RunPod's persistent volume and this Windows box are
                                                       # both already local disk, no separate copy needed.
OUT_BASE = os.path.join(ROOT, 'sft_outputs')          # checkpoints/logs -- persist across sessions
os.makedirs(LOCAL_DIR, exist_ok=True)
os.makedirs(OUT_BASE, exist_ok=True)

if ON_RUNPOD:
    import subprocess
    RCLONE_REMOTE = os.environ.get('FHIRSQL_RCLONE_REMOTE', 'gdrive:projects/FHIRSQL')
    if shutil.which('rclone') is None:
        print('[runpod] rclone not found -- installing...')
        subprocess.run('curl https://rclone.org/install.sh | sudo bash', shell=True, check=True)
    remote_name = RCLONE_REMOTE.split(':')[0] + ':'
    remotes = subprocess.run(['rclone', 'listremotes'], capture_output=True, text=True).stdout
    if remote_name not in remotes:
        raise RuntimeError(
            f"rclone has no '{remote_name}' remote configured. This can't be set up non-"
            f"interactively (Google OAuth needs a browser step) -- run `rclone config` once on "
            f"this pod, or copy an already-authorized rclone.conf to ~/.config/rclone/rclone.conf, "
            f"then re-run this cell. See https://rclone.org/drive/ for the one-time setup."
        )
    print(f'[runpod] syncing {RCLONE_REMOTE} -> {ROOT} (skips files that already match -- fast on repeat runs)...')
    subprocess.run(['rclone', 'copy', RCLONE_REMOTE, ROOT, '--progress'], check=True)


def push_to_drive_backup():
    '''Not called automatically -- checkpoints on the RunPod persistent volume are already
    durable, this is purely optional convenience/off-volume backup. Call manually whenever wanted.'''
    if not ON_RUNPOD:
        print('push_to_drive_backup() is only meaningful on RunPod (ROOT already IS the Drive-backed store on Colab)')
        return
    import subprocess
    subprocess.run(['rclone', 'copy', ROOT, os.environ.get('FHIRSQL_RCLONE_REMOTE', 'gdrive:projects/FHIRSQL'), '--progress'], check=True)

DRIVE_JSONL = os.path.join(ROOT, 'data', 'training', 'sft_final_plan.jsonl')
DRIVE_SCHEMA = os.path.join(ROOT, 'schema', 'schema.sql')
DRIVE_DUCKDB = os.path.join(ROOT, 'data', 'train.duckdb')   # only copied later if CFG['run_execution_eval']
DRIVE_MODEL_PATH = os.path.join(ROOT, 'Qwen2.5-Coder-14B-Instruct')


def _copy_local(src, dst_name):
    '''Copy once to local session disk -- don't repeatedly read the Drive-mounted copy directly,
    FUSE read latency is real for many-small-reads access patterns (see METHODOLOGY_LOG.md,
    "Storage and infrastructure").'''
    dst = os.path.join(LOCAL_DIR, dst_name)
    if not os.path.exists(dst):
        shutil.copy2(src, dst)
    return dst


def _copy_local_dir(src, dst_name):
    dst = os.path.join(LOCAL_DIR, dst_name)
    if not os.path.exists(dst):
        shutil.copytree(src, dst)
    return dst


LOCAL_JSONL = _copy_local(DRIVE_JSONL, 'sft_final_plan.jsonl')
LOCAL_SCHEMA = _copy_local(DRIVE_SCHEMA, 'schema.sql')
LOCAL_MODEL_PATH = _copy_local_dir(DRIVE_MODEL_PATH, 'Qwen2.5-Coder-14B-Instruct')

print('on_colab =', ON_COLAB, '| on_runpod =', ON_RUNPOD)
print('ROOT     =', ROOT)
print('OUT_BASE =', OUT_BASE)
print('LOCAL_JSONL      =', LOCAL_JSONL)
print('LOCAL_SCHEMA     =', LOCAL_SCHEMA)
print('LOCAL_MODEL_PATH =', LOCAL_MODEL_PATH)

In [ ]:
# ==== imports ====
import json, random, time, math, sys, functools, contextlib, statistics, re
from collections import Counter

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import get_peft_model, LoraConfig, TaskType, prepare_model_for_kbit_training, PeftModel

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
USE_AMP = (DEVICE == 'cuda')
AMP_DTYPE = torch.bfloat16


def amp_ctx():
    return torch.autocast('cuda', dtype=AMP_DTYPE) if USE_AMP else contextlib.nullcontext()


if DEVICE == 'cuda':
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    # NOT setting cudnn.benchmark=True here (unlike the reference notebook): that reference had
    # fixed-shape spectrogram inputs, so autotuning paid for itself once. Here batches are
    # dynamically padded to the batch's own longest sequence, so shapes vary every step --
    # benchmark mode would keep re-tuning kernels rather than reusing a cached plan.
    print('GPU:', torch.cuda.get_device_name(0),
          f'| {torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB | AMP bf16 ON | TF32 ON')
print('torch', torch.__version__, '| device', DEVICE)

In [ ]:
# ==== CONFIG ====
CFG = dict(
    model_name=LOCAL_MODEL_PATH,   # local snapshot, not the Hub id -- see setup cell
    quant_type='nf4',

    # DoRA -- see METHODOLOGY_LOG.md "Supervised fine-tuning setup" for the rationale behind all
    # of these (all-linear-layer target modules, r=16/alpha=32 as a dataset-size-appropriate
    # starting point, bias frozen).
    dora_rank=16, dora_alpha=32, dora_dropout=0.05, dora_bias='none',
    dora_target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],

    max_seq_len=4096,          # prompt (~1,200-1,500 tokens) + up to 2048 completion tokens needs
                                # headroom; a char-length-based estimate, verify against cell 10's
                                # token-length probe on the real tokenizer.
    dev_fraction=0.08, test_fraction=0.08, split_seed=20260804,   # grouped 3-way split, fixed across all seeds

    micro_batch_size=14, grad_accum_steps=1,  # empirically stable across repeated real runs on this model/GPU combination
    use_torch_compile=False,  # opt-in -- torch.compile via Triton/TorchInductor can speed up training, but this codebase pads each batch to its own longest sequence (see make_loader), so shapes vary every step; full static compilation would recompile on every new shape, which can cost more than it saves. See _maybe_compile below for the dynamic=True mitigation.
    num_workers=2,

    lr=2e-4,            # LoRA/DoRA adapters tolerate (and typically want) a higher LR than full fine-tuning
    weight_decay=0.0,   # adapter-only training; not decaying an already rank-constrained update
    warmup_ratio=0.03,
    max_epochs=3,        # paired with early-stopping logic in SFTTrainer.train(): trains epoch 1
                          # and 2 unconditionally, then only attempts epoch 3 if epoch 2's dev loss
                          # improved on epoch 1's by at least sft_early_stop_min_relative_improvement.
    sft_early_stop_min_relative_improvement=0.05,  # relative val_loss improvement (epoch1->epoch2)
                          # below which epoch 3 is skipped as not worth the compute.

    seeds=[42, 43, 44],   # same seed values as the reference notebook, for no reason other than continuity

    eval_gen_sample_size=60, eval_gen_every_n_epochs=1, max_new_tokens_eval=2048,
    eval_gen_batch_size=14,     # prompts batched per generate() call in evaluate_generation --
                                # reuses micro_batch_size as an already-proven-safe bound; no
                                # backward pass here so this could likely go higher, but this
                                # avoids a separate probe
    efficiency_min_timing_ms=0.5,  # floor under which wall-clock timing is dominated by measurement
                                    # noise -- see gen_efficiency_speedup below
    run_execution_eval=True,   # set False to skip copying/querying train.duckdb during eval
)
print(json.dumps(CFG, indent=2))

## Model wrapper

In [ ]:
def _pick_attn_implementation():
    '''Flash Attention 2 needs the flash-attn package AND Ampere+ (compute capability >= 8.0) --
    neither is guaranteed on every GPU this notebook might run on, so this probes rather than
    assumes, falling back to
    PyTorch's own SDPA (already a fused, efficient attention kernel, just without FA2's specific
    memory/speed profile) instead of failing outright if FA2 isn't available or isn't supported.'''
    if not torch.cuda.is_available():
        return 'sdpa'
    try:
        import flash_attn  # noqa: F401
    except ImportError:
        return 'sdpa'
    major, _minor = torch.cuda.get_device_capability()
    if major < 8:
        return 'sdpa'
    return 'flash_attention_2'


class FHIRSQLLLM:
    '''Base model + DoRA adapter wrapper.'''

    def __init__(self, cfg):
        self.model_name = cfg.get('model_name', 'Qwen/Qwen2.5-Coder-14B-Instruct')
        self.quant_type = cfg.get('quant_type', 'nf4')
        self.dtype = torch.bfloat16
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.dora_rank = cfg.get('dora_rank', 16)
        self.dora_alpha = cfg.get('dora_alpha', 32)
        self.dora_dropout = cfg.get('dora_dropout', 0.05)
        self.dora_bias = cfg.get('dora_bias', 'none')
        self.dora_target_modules = cfg.get(
            'dora_target_modules',
            ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
        )

    def build(self):
        tokenizer = self._load_tokenizer()
        dora_model = get_peft_model(self._load_base_model(), self._get_dora())
        self.inspect_trainable_parameters(dora_model)
        return dora_model, tokenizer

    def _load_tokenizer(self):
        tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        tokenizer.padding_side = 'right'   # training default; generation-eval flips this temporarily
        if tokenizer.pad_token_id is None:
            tokenizer.pad_token = tokenizer.eos_token
        return tokenizer

    def _load_base_model(self):
        attn_impl = _pick_attn_implementation()
        model = AutoModelForCausalLM.from_pretrained(
            self.model_name, dtype=self.dtype, quantization_config=self._get_quantization_config(),
            attn_implementation=attn_impl,
        )
        print(f'attn_implementation = {attn_impl}')
        model.config.use_cache = False
        return prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

    def _get_quantization_config(self):
        return BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_compute_dtype=self.dtype,
            bnb_4bit_use_double_quant=True, bnb_4bit_quant_type=self.quant_type,
        )

    def _get_dora(self):
        return LoraConfig(
            r=self.dora_rank, lora_alpha=self.dora_alpha, bias=self.dora_bias,
            lora_dropout=self.dora_dropout, use_dora=True,
            target_modules=self.dora_target_modules, task_type=TaskType.CAUSAL_LM,
        )

    def inspect_trainable_parameters(self, model):
        total = sum(p.numel() for p in model.parameters())
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        names = [n for n, p in model.named_parameters() if p.requires_grad]
        print(f'total params: {total:,} | trainable: {trainable:,} ({100*trainable/total:.3f}%)')
        assert trainable > 0
        assert any('lora_A' in n for n in names)
        assert any('lora_B' in n for n in names)
        assert any('lora_magnitude_vector' in n for n in names)   # DoRA-specific -- confirms use_dora actually took effect

## Data loading and prompt format

Schema DDL is extracted from `schema/schema.sql`, dropping the human-facing changelog header
(everything before the first `CREATE TABLE`) -- the model only needs the DDL plus its inline
column comments (several encode real signal, e.g. the note that `encounter` only ever has one
participant, which is exactly why physician-role-disambiguation questions are `UNANSWERABLE`).
The system prompt explicitly instructs the abstention behavior, since 8.6% of the training set's
gold target is the literal string `UNANSWERABLE`, not SQL.

In [ ]:
ABSTENTION_TOKEN = 'UNANSWERABLE'

SYSTEM_PROMPT_TEMPLATE = '''You are a clinical data analyst who translates natural-language hospital \
questions into DuckDB SQL, run against the schema below, via an explicit query plan first.

Output exactly two parts, in this order, and nothing else:
1. A JSON object describing the query plan: which clinical concepts the question refers to (and \
whether each needs a terminology lookup against the schema's `valuesets` table), what \
additional tables must be joined and why, what filters apply, and what the final aggregation \
computes.
2. The compiled SQL statement for that plan, in a fenced code block:
```sql
<the SQL statement>
```

If the question cannot be answered from this schema -- the data it needs genuinely doesn't \
exist here -- the plan should be {{"abstain": true}}, and the fenced SQL block should contain \
exactly the single word: {abstention_token}

Do not guess or approximate an answer using unrelated columns when the real field is absent.

Schema:
{schema_ddl}'''


_SQL_FENCE_RE = re.compile(r"```sql\s*\n(.*?)\n```", re.IGNORECASE | re.DOTALL)


def extract_sql_from_completion(text):
    '''Pulls the SQL out of a `plan JSON` + fenced-```sql-block completion (mirrors rl_train.ipynb).
    Takes the LAST matching fence, not the first. Returns None (not a
    crash, not an empty string) when no fence is found at all.'''
    matches = _SQL_FENCE_RE.findall(text)
    if not matches:
        return None
    return matches[-1].strip()


def load_schema_ddl(schema_path):
    text = open(schema_path, encoding='utf-8').read()
    idx = text.index('CREATE TABLE')   # drop the changelog header -- see markdown cell above
    return text[idx:].strip()


def build_messages(row, schema_ddl):
    system = SYSTEM_PROMPT_TEMPLATE.format(abstention_token=ABSTENTION_TOKEN, schema_ddl=schema_ddl)
    return [
        {'role': 'system', 'content': system},
        {'role': 'user', 'content': row['question']},
        {'role': 'assistant', 'content': row['target']},
    ]


def chat_template_ids(tokenizer, messages, add_generation_prompt):
    '''Renders the chat template to text, then tokenizes that text directly, instead of relying
    on apply_chat_template(tokenize=True)'s return type -- that return type (plain list of ids vs
    a dict/BatchEncoding) is not consistent across transformers versions. tokenize=False + a
    separate tokenizer() call is unambiguous: the template text already contains any needed
    special-token markup as literal substrings, so add_special_tokens=False here.'''
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=add_generation_prompt)
    return tokenizer(text, add_special_tokens=False)['input_ids']


_OPERATIONAL_IDS = {
    'doctor_top5_by_encounter_type', 'doctor_top5_prescribers', 'doctor_top5_by_condition_diagnosed',
    'bed_occupancy_at_date', 'bed_occupancy_by_month', 'avg_length_of_stay',
    'current_inpatient_census_by_month',
}
_QUALITY_KPI_IDS = {
    'condition_by_race', 'condition_by_ethnicity', 'physician_monthly_case_volume',
    'readmission_rate_30day', 'mortality_review_monthly', 'fall_risk_screening_monthly',
    'patients_by_race', 'patients_by_ethnicity', 'patients_by_state',
    'imaging_volume_by_month', 'imaging_volume_by_modality',
}


def archetype_category(archetype_id):
    if archetype_id.startswith('unans_'):
        return 'unanswerable'
    if archetype_id.startswith('reg_'):
        return 'regulatory'
    if archetype_id in _OPERATIONAL_IDS:
        return 'operational'
    if archetype_id in _QUALITY_KPI_IDS:
        return 'quality_kpi_demographics'
    return 'main'


def instance_key(row):
    # Paraphrases of the SAME underlying SQL instance must land on the same side of the split --
    # otherwise dev "accuracy" would partly measure memorized phrasing, not generalization.
    return f"{row['archetype_id']}|{row.get('concept_display')}|{row['gold_sql']}"


def grouped_split(rows, dev_fraction, test_fraction, seed):
    '''Three-way split, grouped by underlying SQL instance (see instance_key) so paraphrases of
    the same query never span two sides, and stratified by (tier, archetype_category) so small
    categories -- e.g. the 31-instance regulatory archetypes, or the 9 fixed-abstention groups --
    get proportional dev/test representation instead of risking exclusion by chance under one
    pooled random split. dev is touched DURING training (per-epoch checkpoint selection); test is
    never touched until the dedicated final-evaluation section below.'''
    groups = {}
    stratum_of = {}
    for r in rows:
        k = instance_key(r)
        groups.setdefault(k, []).append(r)
        stratum_of[k] = (r['tier'], archetype_category(r['archetype_id']))

    by_stratum = {}
    for k, stratum in stratum_of.items():
        by_stratum.setdefault(stratum, []).append(k)

    rng = random.Random(seed)
    dev_keys, test_keys = set(), set()
    for stratum in sorted(by_stratum.keys()):
        keys = sorted(by_stratum[stratum])
        rng.shuffle(keys)
        n = len(keys)
        if n < 3:
            continue   # too few groups in this (tier, category) to safely carve out dev/test -- keep all in train
        n_dev = max(1, round(n * dev_fraction))
        n_test = max(1, round(n * test_fraction))
        while n_dev + n_test > n - 1:   # never remove an entire small stratum from train
            if n_dev >= n_test:
                n_dev -= 1
            else:
                n_test -= 1
        dev_keys.update(keys[:n_dev])
        test_keys.update(keys[n_dev:n_dev + n_test])

    train_rows, dev_rows, test_rows = [], [], []
    for k, rs in groups.items():
        if k in dev_keys:
            dev_rows.extend(rs)
        elif k in test_keys:
            test_rows.extend(rs)
        else:
            train_rows.extend(rs)
    return train_rows, dev_rows, test_rows


rows = [json.loads(l) for l in open(LOCAL_JSONL, encoding='utf-8')]
schema_ddl = load_schema_ddl(LOCAL_SCHEMA)
train_rows, dev_rows, test_rows = grouped_split(rows, CFG['dev_fraction'], CFG['test_fraction'], CFG['split_seed'])

train_keys = {instance_key(r) for r in train_rows}
dev_keys = {instance_key(r) for r in dev_rows}
test_keys = {instance_key(r) for r in test_rows}
assert train_keys.isdisjoint(dev_keys), 'train/dev instance-key leakage'
assert train_keys.isdisjoint(test_keys), 'train/test instance-key leakage'
assert dev_keys.isdisjoint(test_keys), 'dev/test instance-key leakage'

print(f'rows: {len(rows)} | train: {len(train_rows)} | dev: {len(dev_rows)} ({len(dev_rows)/len(rows)*100:.1f}%) '
      f'| test: {len(test_rows)} ({len(test_rows)/len(rows)*100:.1f}%)')
print('train tier counts:', dict(sorted(Counter(r['tier'] for r in train_rows).items())))
print('dev   tier counts:', dict(sorted(Counter(r['tier'] for r in dev_rows).items())))
print('test  tier counts:', dict(sorted(Counter(r['tier'] for r in test_rows).items())))
print('train category counts:', dict(sorted(Counter(archetype_category(r['archetype_id']) for r in train_rows).items())))
print('dev   category counts:', dict(sorted(Counter(archetype_category(r['archetype_id']) for r in dev_rows).items())))
print('test  category counts:', dict(sorted(Counter(archetype_category(r['archetype_id']) for r in test_rows).items())))
print('schema_ddl chars:', len(schema_ddl), '| first line:', schema_ddl.splitlines()[0])

# Fixed generation-eval subsamples -- the SAME dev rows are scored for the dev baseline and every
# training epoch/seed; the SAME test rows are scored for the final test baseline and every seed's
# best checkpoint -- so comparisons reflect the model, not which rows got sampled.
GEN_EVAL_SAMPLE = random.Random(CFG['split_seed']).sample(
    dev_rows, min(CFG['eval_gen_sample_size'], len(dev_rows))
)
TEST_GEN_SAMPLE = random.Random(CFG['split_seed']).sample(
    test_rows, min(CFG['eval_gen_sample_size'], len(test_rows))
)
print(f'fixed generation-eval samples: dev={len(GEN_EVAL_SAMPLE)} rows, test={len(TEST_GEN_SAMPLE)} rows')

In [ ]:
# ==== sanity: token-length stats (pick max_seq_len) + chat-template prefix invariant ====
# Label masking below assumes apply_chat_template(messages[:-1], add_generation_prompt=True) is a
# strict token-level PREFIX of apply_chat_template(messages, add_generation_prompt=False). This is
# true for ChatML-style templates (Qwen included) but is verified here directly rather than assumed,
# consistent with this project's validate-empirically discipline.
_tok_probe = AutoTokenizer.from_pretrained(CFG['model_name'])

sample_for_check = random.sample(rows, min(300, len(rows)))
lens, mismatches = [], 0
for r in sample_for_check:
    msgs = build_messages(r, schema_ddl)
    p = chat_template_ids(_tok_probe, msgs[:-1], add_generation_prompt=True)
    f = chat_template_ids(_tok_probe, msgs, add_generation_prompt=False)
    lens.append(len(f))
    if f[:len(p)] != p:
        mismatches += 1

lens = np.array(lens)
print(f'token length over {len(lens)} sampled rows: mean={lens.mean():.0f} p50={np.percentile(lens,50):.0f} '
      f'p95={np.percentile(lens,95):.0f} max={lens.max()}')
print(f'chat-template prefix invariant: {len(sample_for_check)-mismatches}/{len(sample_for_check)} rows match')
assert mismatches == 0, (
    'prompt-prefix is not a prefix of the full conversation for some rows -- label masking in '
    'SFTDataset would be wrong for those rows. Fix before training, do not proceed on a warning.'
)
print(f"\nCFG['max_seq_len'] is currently {CFG['max_seq_len']} -- adjust above if p95/max here says otherwise.")

## Dataset and loaders

One item = one training row -> a tokenized chat-template conversation with `labels` masked
(`-100`) over everything except the assistant's SQL/`UNANSWERABLE` tokens, so loss is computed
only on what the model must actually produce. Dynamic padding per batch (not a fixed max length)
via a custom `collate_fn`.

In [ ]:
class SFTDataset(Dataset):
    '''labels mask everything up to (and including) the assistant generation-prompt tag with
    -100, so loss is computed only on the SQL/UNANSWERABLE tokens the model must produce.'''

    def __init__(self, rows, tokenizer, schema_ddl, max_seq_len):
        self.rows = rows
        self.tokenizer = tokenizer
        self.schema_ddl = schema_ddl
        self.max_seq_len = max_seq_len

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        row = self.rows[i]
        messages = build_messages(row, self.schema_ddl)
        prompt_ids = chat_template_ids(self.tokenizer, messages[:-1], add_generation_prompt=True)
        full_ids = chat_template_ids(self.tokenizer, messages, add_generation_prompt=False)
        full_ids = full_ids[: self.max_seq_len]
        prompt_len = min(len(prompt_ids), len(full_ids))
        assert prompt_len < len(full_ids), (
            f'row {i} (archetype={row["archetype_id"]!r}): truncation to max_seq_len='
            f'{self.max_seq_len} left no room for the answer (prompt alone is {len(prompt_ids)} '
            f'tokens) -- every label would be -100, producing NaN loss. Raise max_seq_len.'
        )
        labels = [-100] * prompt_len + full_ids[prompt_len:]
        return {'input_ids': full_ids, 'labels': labels}


def collate_fn(batch, pad_token_id):
    max_len = max(len(b['input_ids']) for b in batch)
    input_ids, labels, attn = [], [], []
    for b in batch:
        pad = max_len - len(b['input_ids'])
        input_ids.append(b['input_ids'] + [pad_token_id] * pad)
        labels.append(b['labels'] + [-100] * pad)
        attn.append([1] * len(b['input_ids']) + [0] * pad)
    return {
        'input_ids': torch.tensor(input_ids, dtype=torch.long),
        'labels': torch.tensor(labels, dtype=torch.long),
        'attention_mask': torch.tensor(attn, dtype=torch.long),
    }


def _worker_init(_wid):
    try:
        torch.set_num_threads(1)
    except Exception:
        pass


def _loader_kwargs(cfg):
    nw = cfg['num_workers']
    kw = dict(num_workers=nw, pin_memory=(nw > 0 and torch.cuda.is_available()))
    if nw > 0:
        kw.update(persistent_workers=True, worker_init_fn=_worker_init, prefetch_factor=2)
    return kw


def make_loader(rows_, tokenizer, schema_ddl_, cfg, shuffle):
    ds = SFTDataset(rows_, tokenizer, schema_ddl_, cfg['max_seq_len'])
    collate = functools.partial(collate_fn, pad_token_id=tokenizer.pad_token_id)
    return DataLoader(ds, batch_size=cfg['micro_batch_size'], shuffle=shuffle, drop_last=shuffle,
                       collate_fn=collate, **_loader_kwargs(cfg))

## Sanity: overfit a single batch, and a quick rank check

Trains only on a handful of fixed examples for many steps -- a correctly-wired pipeline should
drive loss toward zero and token accuracy toward 1.0 on data it's allowed to memorize. If it
doesn't, something in the pipeline (label masking, gradient flow, tokenization) is broken, and
that's worth catching here rather than partway through the real multi-hour run.

Runs only at the configured `CFG['dora_rank']` -- rank=16/alpha=32 already found stable across
repeated real runs, no need to re-sweep candidate ranks here.

In [ ]:
def overfit_one_batch(model, tokenizer, overfit_rows, schema_ddl_, cfg, n_steps=25, lr=1e-3):
    '''Trains `model` (already peft-wrapped) on a handful of fixed examples for n_steps, using
    the same micro-batch + gradient-accumulation mechanics as SFTTrainer._train_epoch. Returns
    loss/token-accuracy before and after, on that same fixed batch.'''
    trainable = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(trainable, lr=lr)

    ds = SFTDataset(overfit_rows, tokenizer, schema_ddl_, cfg['max_seq_len'])
    items = [ds[i] for i in range(len(ds))]

    losses = []
    model.train()
    looper = tqdm(range(n_steps), desc='overfit steps')
    for step in looper:
        opt.zero_grad(set_to_none=True)
        step_loss = 0.0
        for item in items:
            batch = collate_fn([item], pad_token_id=tokenizer.pad_token_id)
            with amp_ctx():
                out = model(input_ids=batch['input_ids'].to(DEVICE),
                            attention_mask=batch['attention_mask'].to(DEVICE),
                            labels=batch['labels'].to(DEVICE))
                (out.loss / len(items)).backward()
            step_loss += out.loss.item()
        torch.nn.utils.clip_grad_norm_(trainable, 1.0)
        opt.step()
        step_loss /= len(items)
        losses.append(step_loss)
        looper.set_postfix(loss=f'{step_loss:.3f}')

    model.eval()
    total_loss, total_correct, total_tokens = 0.0, 0, 0
    with torch.no_grad():
        for item in items:
            batch = collate_fn([item], pad_token_id=tokenizer.pad_token_id)
            with amp_ctx():
                out = model(input_ids=batch['input_ids'].to(DEVICE),
                            attention_mask=batch['attention_mask'].to(DEVICE),
                            labels=batch['labels'].to(DEVICE))
            total_loss += out.loss.item()
            preds = out.logits[:, :-1, :].argmax(-1)
            tgt = batch['labels'][:, 1:].to(DEVICE)
            mask = tgt != -100
            total_correct += (preds[mask] == tgt[mask]).sum().item()
            total_tokens += mask.sum().item()

    final_loss = total_loss / len(items)
    return {
        'first_loss': losses[0], 'first_perplexity': math.exp(losses[0]),
        'min_loss': min(losses),
        'final_loss': final_loss, 'final_perplexity': math.exp(final_loss),
        'final_token_acc': total_correct / max(1, total_tokens),
        'losses': losses,
    }


OVERFIT_ROWS = train_rows[:4]
_overfit_wrapper = FHIRSQLLLM(CFG)
_overfit_tokenizer = _overfit_wrapper._load_tokenizer()
_overfit_base_model = _overfit_wrapper._load_base_model()   # loaded once, reused across every rank below

CANDIDATE_RANKS = [CFG['dora_rank']]   # single rank only -- see cell above
overfit_results = {}
for r in CANDIDATE_RANKS:
    print(f'--- overfit-one-batch: dora_rank={r} ---')
    rank_wrapper = FHIRSQLLLM(dict(CFG, dora_rank=r))
    model = get_peft_model(_overfit_base_model, rank_wrapper._get_dora())
    model.to(DEVICE)
    res = overfit_one_batch(model, _overfit_tokenizer, OVERFIT_ROWS, schema_ddl, CFG)
    overfit_results[r] = res
    print(f"  first_loss={res['first_loss']:.4f} -> final_loss={res['final_loss']:.4f} "
          f"(min {res['min_loss']:.4f}) | ppl {res['first_perplexity']:.2f} -> {res['final_perplexity']:.2f} "
          f"| final_token_acc={res['final_token_acc']:.3f}")
    _overfit_base_model = model.unload()
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()

print()
print(f"{'rank':<6} {'first_loss':<12} {'final_loss':<12} {'min_loss':<10} {'final_ppl':<10} {'final_token_acc':<16}")
for r in CANDIDATE_RANKS:
    res = overfit_results[r]
    print(f"{r:<6} {res['first_loss']:<12.4f} {res['final_loss']:<12.4f} "
          f"{res['min_loss']:<10.4f} {res['final_perplexity']:<10.2f} {res['final_token_acc']:<16.3f}")

_configured = overfit_results[CFG['dora_rank']]
assert _configured['final_loss'] < _configured['first_loss'] * 0.1, (
    f"loss did not collapse on a single memorizable batch at the configured rank "
    f"({CFG['dora_rank']}) -- first={_configured['first_loss']:.4f} "
    f"final={_configured['final_loss']:.4f}. Something in the pipeline (label masking, "
    f"gradient flow, learning rate) is likely broken -- fix before the real training run."
)
print(f"\nconfigured rank {CFG['dora_rank']} passes the overfit check.")

del _overfit_base_model
if DEVICE == 'cuda':
    torch.cuda.empty_cache()

## Logging, checkpointing, and evaluation metrics

`TeeFile` and `set_seed` reused verbatim from the reference pipeline. Two eval paths: a fast
per-epoch loss/token-accuracy pass (drives best-checkpoint selection) and a slower periodic
generation-based spot-check (exact-match / execution-match / abstention precision-recall).

In [ ]:
class TeeFile:
    '''Write to stdout AND a logfile simultaneously.'''
    def __init__(self, *targets):
        self.files, self._opened = [], []
        for t in targets:
            if isinstance(t, str):
                f = open(t, 'a', buffering=1); self.files.append(f); self._opened.append(f)
            else:
                self.files.append(t)
    def write(self, data):
        for f in self.files:
            try: f.write(data); f.flush()
            except Exception: pass
    def flush(self):
        for f in self.files:
            try: f.flush()
            except Exception: pass
    def close(self):
        for f in self._opened:
            try: f.close()
            except Exception: pass


def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


def _maybe_compile(model, cfg):
    '''torch.compile (Triton/TorchInductor) -- opt-in via CFG['use_torch_compile'], default
    False. This codebase deliberately pads each batch to its own longest sequence rather than a
    fixed length (see make_loader/collate_fn), so shapes vary every step; a full static compile
    would recompile on every new shape, which can cost more wall-clock than it saves.
    dynamic=True compiles a shape-flexible graph instead, trading some peak speedup for
    robustness to that variability -- still experimental for a 4-bit-quantized + PEFT model
    (bitsandbytes' custom autograd ops are a known source of graph breaks that can silently
    reduce or eliminate the speedup), so test on a handful of real steps before trusting it for
    a full run rather than assuming it helps.'''
    if not cfg.get('use_torch_compile', False):
        return model
    if not (torch.cuda.is_available() and hasattr(torch, 'compile')):
        return model
    return torch.compile(model, dynamic=True)


def _atomic_save_adapter(model, path, retries=6, delay=2.0):
    '''Save a PEFT adapter to `path` via temp-dir + swap, retrying on transient Drive/OS locks.
    Never raises -- a failed checkpoint save shouldn't crash a multi-hour run.'''
    tmp = path + f'.tmp{os.getpid()}'
    for attempt in range(retries):
        try:
            if os.path.exists(tmp):
                shutil.rmtree(tmp)
            model.save_pretrained(tmp)
            if os.path.exists(path):
                shutil.rmtree(path)
            os.replace(tmp, path)
            return
        except (RuntimeError, OSError, PermissionError) as e:
            if attempt == retries - 1:
                print(f'  [warn] adapter save failed after {retries} tries ({e}); continuing')
                return
            time.sleep(delay)


def _atomic_torch_save(obj, path, retries=6, delay=1.5):
    d = os.path.dirname(path); base = os.path.basename(path)
    for attempt in range(retries):
        tmp = os.path.join(d, f'.{base}.tmp{os.getpid()}')
        try:
            torch.save(obj, tmp); os.replace(tmp, path); return
        except (RuntimeError, OSError, PermissionError) as e:
            try:
                if os.path.exists(tmp): os.remove(tmp)
            except Exception: pass
            if attempt == retries - 1:
                print(f'  [warn] state save failed after {retries} tries ({e}); continuing'); return
            time.sleep(delay)

In [ ]:
@torch.no_grad()
def evaluate_loss_and_token_acc(model, loader, amp_ctx_fn):
    model.eval()
    total_loss, total_tokens, correct_tokens, n_batches = 0.0, 0, 0, 0
    for batch in loader:
        input_ids = batch['input_ids'].to(DEVICE)
        labels = batch['labels'].to(DEVICE)
        attn = batch['attention_mask'].to(DEVICE)
        with amp_ctx_fn():
            out = model(input_ids=input_ids, attention_mask=attn, labels=labels)
        total_loss += out.loss.item(); n_batches += 1
        preds = out.logits[:, :-1, :].argmax(-1)
        tgt = labels[:, 1:]
        mask = tgt != -100
        correct_tokens += (preds[mask] == tgt[mask]).sum().item()
        total_tokens += mask.sum().item()
    mean_loss = total_loss / max(1, n_batches)
    return {'loss': mean_loss, 'perplexity': math.exp(mean_loss),
            'token_acc': correct_tokens / max(1, total_tokens)}


def _round_val(v, ndigits=2):
    # Float-tolerant comparison: DuckDB's parallel aggregation reorders
    # floating-point summation non-associatively across threads, so byte-identical SQL run twice
    # can return values differing at the 10th-15th decimal place -- confirmed directly by
    # repeated execution of real gold SQL (61/80 AVG(valueQuantity) queries differed across 5
    # runs before this fix). Gold SQL now rounds AVG(valueQuantity) to 4 decimals at the source
    # (see archetypes.py), which fixes ~95% of cases, but GROUP BY-year aggregates can still
    # occasionally straddle a rounding boundary (e.g. 4.1260 vs 4.1261) since that's a genuine
    # boundary-flip, not raw jitter, and no amount of added SQL-level decimal precision
    # eliminates a boundary problem -- only a coarser comparison tolerance does. Rounding to 2
    # decimals here gives 100x margin over the observed ~0.0001 residual, applied uniformly (not
    # just to the known-affected archetypes) so any other float-producing SQL is covered too.
    if isinstance(v, float):
        return round(v, ndigits)
    return v


_HARDCODED_TERMINOLOGY_RE = re.compile(
    r"\b(?:code|system|type_code|type_system|vaccineCode|vaccineCode_system)\s*=\s*'[^']*'",
    re.IGNORECASE,
)


def has_hardcoded_terminology(sql):
    '''Mirrors rl_train.ipynb's detector of the same name (see there for the full rationale):
    True if `sql` compares a terminology code/system column directly to a quoted literal
    instead of via the lookup-CTE join pattern. Diagnostic-only here (SFT has no reward to
    gate), tracked via gen_hardcoded_rate below.'''
    return bool(_HARDCODED_TERMINOLOGY_RE.search(sql))


def _exec_rows(con, sql):
    # Plain sorted() crashes ('<' not supported between float and NoneType) the moment a result
    # column mixes NULL and non-NULL values across rows -- e.g. AVG() returning NULL for one
    # group but a real number for another. Both callers of this function catch exceptions and
    # score the query as wrong (0.0 / no match) on any error, so this was silently mis-scoring
    # otherwise-correct queries rather than crashing loudly. Sorting on (is_none, value) per
    # column keeps None grouped and orderable without ever comparing it to a real value.
    rows = [tuple(_round_val(v) for v in row) for row in con.execute(sql).fetchall()]
    return sorted(rows, key=lambda row: tuple((v is None, v) for v in row))


@torch.no_grad()
def evaluate_generation(model, tokenizer, sample_rows, schema_ddl_, cfg, duckdb_con=None):
    '''Spot-check via greedy decoding: exact SQL-text match, execution-result match against
    train.duckdb (answerable rows only, order-insensitive), abstention precision/recall, and
    gen_efficiency_speedup. Abstention is scored separately from SQL correctness because a correct
    refusal and a confident wrong query are different failure modes (see METHODOLOGY_LOG.md,
    "Training-data generation"). `sample_rows` is a fixed, pre-selected subsample (see
    GEN_EVAL_SAMPLE) rather than freshly sampled here -- baseline and every epoch/seed must score
    the identical rows for the comparison to reflect the model, not sampling noise.

    Batched: multiple prompts per generate() call
    instead of one at a time -- was fine at eval_gen_sample_size=60 but no reason to pay the
    per-call overhead 60 times when one batched call does the same work.

    gen_efficiency_speedup: median (gold execution
    time / predicted execution time) among rows scored exec-correct -- NaN if none were correct.
    Measured cache-fair: an untimed warm-up pass on both queries first (equalizes DuckDB
    buffer-pool state), then a timed second execution of each with randomized order, so this
    metric isn't systematically biased toward whichever query happens to run second. This SFT
    notebook has no RL reward to explain, but this metric lets the ablation table in rl_train.ipynb
    show a real SFT-only baseline instead of NaN for every frozen/SFT-only row.'''
    model.eval()
    prev_padding_side = tokenizer.padding_side
    tokenizer.padding_side = 'left'   # batched/greedy generation needs left-padding, unlike training
    batch_size = cfg.get('eval_gen_batch_size', cfg['micro_batch_size'])

    n_exact = 0
    n_hardcoded = 0   # see has_hardcoded_terminology below -- diagnostic only here
    abst_tp = abst_fp = abst_fn = abst_tn = 0
    exec_match = exec_total = 0
    speedups = []

    for start in range(0, len(sample_rows), batch_size):
        batch_rows = sample_rows[start:start + batch_size]
        prompt_texts = []
        for r in batch_rows:
            messages = build_messages(r, schema_ddl_)[:-1]
            prompt_texts.append(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))

        enc = tokenizer(prompt_texts, return_tensors='pt', padding=True, truncation=True,
                         max_length=cfg['max_seq_len'], add_special_tokens=False).to(DEVICE)
        padded_prompt_len = enc['input_ids'].shape[1]

        with torch.no_grad():
            gen = model.generate(**enc, max_new_tokens=cfg['max_new_tokens_eval'], do_sample=False,
                                  pad_token_id=tokenizer.eos_token_id)

        for i, r in enumerate(batch_rows):
            completion = tokenizer.decode(gen[i][padded_prompt_len:], skip_special_tokens=True).strip()
            pred_sql = extract_sql_from_completion(completion)
            pred = (pred_sql or '').strip()   # '' (unparseable completion) falls through to the exec
                                               # attempt below, which fails naturally and is counted,
                                               # rather than being silently excluded from exec_total
            gold = r['gold_sql'].strip()
            is_abst_gold = gold == ABSTENTION_TOKEN
            is_abst_pred = pred == ABSTENTION_TOKEN
            if is_abst_gold and is_abst_pred: abst_tp += 1
            elif is_abst_gold and not is_abst_pred: abst_fn += 1
            elif not is_abst_gold and is_abst_pred: abst_fp += 1
            else: abst_tn += 1
            if not is_abst_gold:
                if pred == gold:
                    n_exact += 1
                if duckdb_con is not None and not is_abst_pred:
                    exec_total += 1
                    try:
                        is_correct = _exec_rows(duckdb_con, pred) == _exec_rows(duckdb_con, gold)
                    except Exception:
                        is_correct = False
                    if is_correct:
                        exec_match += 1
                        if has_hardcoded_terminology(pred):
                            n_hardcoded += 1
                        try:
                            _exec_rows(duckdb_con, pred); _exec_rows(duckdb_con, gold)   # untimed warm-up, both
                            order = ['pred', 'gold'] if random.random() < 0.5 else ['gold', 'pred']
                            timings = {}
                            for which in order:
                                sql = pred if which == 'pred' else gold
                                t0 = time.perf_counter()
                                _exec_rows(duckdb_con, sql)
                                timings[which] = time.perf_counter() - t0
                            floor_s = cfg.get('efficiency_min_timing_ms', 0.5) / 1000
                            speedups.append(max(timings['gold'], floor_s) / max(timings['pred'], floor_s))
                        except Exception:
                            pass   # correctness already scored above; a timing hiccup shouldn't drop the row

    tokenizer.padding_side = prev_padding_side
    n_answerable = sum(1 for r in sample_rows if r['gold_sql'] != ABSTENTION_TOKEN)
    return {
        'gen_exact_match': n_exact / max(1, n_answerable),
        'gen_exec_match': (exec_match / exec_total) if exec_total else float('nan'),
        'gen_abstention_precision': abst_tp / max(1, abst_tp + abst_fp),
        'gen_abstention_recall': abst_tp / max(1, abst_tp + abst_fn),
        'gen_efficiency_speedup': statistics.median(speedups) if speedups else float('nan'),
        'gen_hardcoded_rate': (n_hardcoded / exec_match) if exec_match else float('nan'),
    }

## Baseline: frozen base model, no adapter

Same metrics as a trained checkpoint, computed once with the DoRA adapter disabled
(`model.disable_adapter()`, a PEFT context manager that strips the adapter's effect from every
forward/generate call made inside it) -- i.e. the base model exactly as shipped, with no SFT
applied. This is the reference point every trained checkpoint is measured against to answer "how
much did the SFT actually help." Computed once, not per seed -- the frozen base model's behavior
on a fixed dev subsample is deterministic under greedy decoding (no dropout at eval, no sampling),
so recomputing it per seed would just burn GPU time for an identical answer -- and cached to
`sft_outputs/baseline.json`.

In [ ]:
@torch.no_grad()
def compute_frozen_baseline(model, tokenizer, dev_loader, sample_rows, schema_ddl_, cfg, duckdb_con):
    with model.disable_adapter():
        fast = evaluate_loss_and_token_acc(model, dev_loader, amp_ctx)
        gen = evaluate_generation(model, tokenizer, sample_rows, schema_ddl_, cfg, duckdb_con)
    return {
        'val_loss': fast['loss'], 'val_perplexity': fast['perplexity'], 'val_token_acc': fast['token_acc'],
        'gen_exact_match': gen['gen_exact_match'], 'gen_exec_match': gen['gen_exec_match'],
        'gen_abstention_precision': gen['gen_abstention_precision'],
        'gen_abstention_recall': gen['gen_abstention_recall'],
    }


def _baseline_path():
    return os.path.join(OUT_BASE, 'baseline.json')


def get_or_compute_baseline(model, tokenizer, dev_loader, sample_rows, schema_ddl_, cfg, duckdb_con):
    bp = _baseline_path()
    if os.path.exists(bp):
        return json.load(open(bp))
    print('[baseline] computing frozen base-model metrics (adapter disabled)...')
    t0 = time.time()
    baseline = compute_frozen_baseline(model, tokenizer, dev_loader, sample_rows, schema_ddl_, cfg, duckdb_con)
    baseline['computed_in_seconds'] = time.time() - t0
    json.dump(baseline, open(bp, 'w'), indent=2)
    print(f"[baseline] saved -> {bp} ({baseline['computed_in_seconds']:.0f}s)")
    print(json.dumps(baseline, indent=2))
    return baseline

## Held-out test split: final, untouched evaluation

`dev` is used *during* training for per-epoch checkpoint selection -- reusing it as the final
"how much did SFT help" number would be mildly optimistic, since the best checkpoint was chosen
specifically to minimize loss on exactly that data. `test` (a third grouped split, same
leakage-safe grouping as `dev`, never touched until here) is evaluated exactly once per seed,
after training finishes, using that seed's *best* checkpoint (by dev loss) -- for both the frozen
base model and the trained adapter, on the identical fixed test subsample -- so the "Cross-seed
summary, test split" section further down is the number worth trusting. This is still an
in-corpus split (drawn from the same archetype/concept factory as `train`), not a substitute for
the held-out benchmark -- it measures whether SFT is working at all, not true out-of-corpus
generalization.

Loading a checkpoint here does not reload the 14B base model: `load_adapter` / `set_adapter` /
`delete_adapter` attach and detach just the small adapter tensors on top of the base model
already resident on the GPU.

In [ ]:
@torch.no_grad()
def evaluate_checkpoint_on_test(model, adapter_dir, adapter_name, tokenizer, test_loader,
                                 test_gen_sample, schema_ddl_, cfg, duckdb_con):
    model.load_adapter(adapter_dir, adapter_name=adapter_name)
    prev_adapter = model.active_adapter
    model.set_adapter(adapter_name)
    model.eval()
    fast = evaluate_loss_and_token_acc(model, test_loader, amp_ctx)
    gen = evaluate_generation(model, tokenizer, test_gen_sample, schema_ddl_, cfg, duckdb_con)
    model.set_adapter(prev_adapter)
    model.delete_adapter(adapter_name)
    return {'val_loss': fast['loss'], 'val_perplexity': fast['perplexity'], 'val_token_acc': fast['token_acc'], **gen}


def _test_baseline_path():
    return os.path.join(OUT_BASE, 'test_baseline.json')


def get_or_compute_test_baseline(model, tokenizer, test_loader, test_gen_sample, schema_ddl_, cfg, duckdb_con):
    bp = _test_baseline_path()
    if os.path.exists(bp):
        return json.load(open(bp))
    print('[test baseline] computing frozen base-model metrics on the held-out test split...')
    t0 = time.time()
    baseline = compute_frozen_baseline(model, tokenizer, test_loader, test_gen_sample, schema_ddl_, cfg, duckdb_con)
    baseline['computed_in_seconds'] = time.time() - t0
    json.dump(baseline, open(bp, 'w'), indent=2)
    print(f"[test baseline] saved -> {bp} ({baseline['computed_in_seconds']:.0f}s)")
    print(json.dumps(baseline, indent=2))
    return baseline

## Trainer

One `SFTTrainer` per seed. Resumable: on init, checks `out_dir` for a `last/` adapter checkpoint
(resumes model weights via `PeftModel.from_pretrained`) and `training_state.pt` (resumes
optimizer/scheduler/history/epoch counter) -- re-running after a Colab disconnect continues from
the last completed epoch rather than restarting.

In [ ]:
class SFTTrainer:
    def __init__(self, cfg, seed, train_rows_, dev_rows_, test_rows_, schema_ddl_, out_dir, duckdb_con=None):
        set_seed(seed)
        self.cfg = cfg; self.seed = seed; self.out_dir = out_dir
        self.schema_ddl = schema_ddl_; self.duckdb_con = duckdb_con
        os.makedirs(out_dir, exist_ok=True)
        self._tee = TeeFile(sys.stdout, os.path.join(out_dir, 'train.log'))

        self.model, self.tokenizer = self._build_model_and_tokenizer()
        self.model.to(DEVICE)
        self.model = _maybe_compile(self.model, cfg)
        self.trainable_params = [p for p in self.model.parameters() if p.requires_grad]

        self.train_loader = make_loader(train_rows_, self.tokenizer, schema_ddl_, cfg, shuffle=True)
        self.dev_loader = make_loader(dev_rows_, self.tokenizer, schema_ddl_, cfg, shuffle=False)
        self.test_loader = make_loader(test_rows_, self.tokenizer, schema_ddl_, cfg, shuffle=False)
        self.dev_rows = dev_rows_
        self.test_rows = test_rows_

        self.opt = torch.optim.AdamW(self.trainable_params, lr=cfg['lr'], weight_decay=cfg['weight_decay'])
        steps_per_epoch = math.ceil(len(self.train_loader) / cfg['grad_accum_steps'])
        total_steps = steps_per_epoch * cfg['max_epochs']
        warmup_steps = max(1, int(total_steps * cfg['warmup_ratio']))
        self.sched = torch.optim.lr_scheduler.SequentialLR(
            self.opt,
            schedulers=[
                torch.optim.lr_scheduler.LinearLR(self.opt, start_factor=0.1, total_iters=warmup_steps),
                torch.optim.lr_scheduler.CosineAnnealingLR(self.opt, T_max=max(1, total_steps - warmup_steps)),
            ],
            milestones=[warmup_steps],
        )
        self.history = {'train_loss': [], 'val_loss': [], 'val_perplexity': [], 'val_token_acc': [], 'lr': [],
                         'gen_exact_match': [], 'gen_exec_match': [],
                         'gen_abstention_precision': [], 'gen_abstention_recall': []}
        self.best_val_loss = float('inf'); self.best_epoch = 0; self.start_epoch = 0
        self._maybe_resume_training_state()

    def _build_model_and_tokenizer(self):
        wrapper = FHIRSQLLLM(self.cfg)
        tokenizer = wrapper._load_tokenizer()
        base_model = wrapper._load_base_model()
        last_dir = os.path.join(self.out_dir, 'last')
        if os.path.isdir(last_dir):
            self._log(f'[resume] loading adapter weights from {last_dir}')
            model = PeftModel.from_pretrained(base_model, last_dir, is_trainable=True)
        else:
            model = get_peft_model(base_model, wrapper._get_dora())
        wrapper.inspect_trainable_parameters(model)
        return model, tokenizer

    def _log(self, msg):
        print(msg, file=self._tee)

    def _state_path(self):
        return os.path.join(self.out_dir, 'training_state.pt')

    def _maybe_resume_training_state(self):
        sp = self._state_path()
        if not os.path.exists(sp):
            return
        state = torch.load(sp, map_location=DEVICE)
        self.opt.load_state_dict(state['optimizer'])
        self.sched.load_state_dict(state['scheduler'])
        self.history = state['history']
        self.best_val_loss = state['best_val_loss']; self.best_epoch = state['best_epoch']
        self.start_epoch = state['epoch']
        self._log(f'[resume] training state loaded, resuming from epoch {self.start_epoch + 1}')

    def _save(self, epoch, is_best):
        _atomic_save_adapter(self.model, os.path.join(self.out_dir, 'last'))
        if is_best:
            _atomic_save_adapter(self.model, os.path.join(self.out_dir, 'best'))
        _atomic_torch_save({
            'optimizer': self.opt.state_dict(), 'scheduler': self.sched.state_dict(),
            'epoch': epoch, 'history': self.history,
            'best_val_loss': self.best_val_loss, 'best_epoch': self.best_epoch,
        }, self._state_path())
        json.dump(self.history, open(os.path.join(self.out_dir, 'history.json'), 'w'), indent=2)

    def _train_epoch(self, epoch):
        self.model.train(); running = 0.0; self.opt.zero_grad(set_to_none=True)
        looper = tqdm(self.train_loader, desc=f'[seed{self.seed}] train {epoch+1}', file=self._tee)
        for step, batch in enumerate(looper):
            with amp_ctx():
                out = self.model(input_ids=batch['input_ids'].to(DEVICE),
                                  attention_mask=batch['attention_mask'].to(DEVICE),
                                  labels=batch['labels'].to(DEVICE))
                loss = out.loss / self.cfg['grad_accum_steps']
            loss.backward()
            running += out.loss.item()
            if (step + 1) % self.cfg['grad_accum_steps'] == 0:
                torch.nn.utils.clip_grad_norm_(self.trainable_params, 1.0)
                self.opt.step(); self.sched.step(); self.opt.zero_grad(set_to_none=True)
            if DEVICE == 'cuda' and (step + 1) % 100 == 0:
                torch.cuda.empty_cache()   # releases cached-but-unused blocks back to the driver;
                                           # doesn't fix allocator fragmentation, just caps its growth
            looper.set_postfix(loss=f'{out.loss.item():.3f}')
        return running / max(1, len(self.train_loader))

    def train(self):
        for epoch in range(self.start_epoch, self.cfg['max_epochs']):
            t0 = time.time()
            tr_loss = self._train_epoch(epoch)
            vm = evaluate_loss_and_token_acc(self.model, self.dev_loader, amp_ctx)
            self.history['train_loss'].append(tr_loss)
            self.history['val_loss'].append(vm['loss'])
            self.history['val_perplexity'].append(vm['perplexity'])
            self.history['val_token_acc'].append(vm['token_acc'])
            self.history['lr'].append(self.opt.param_groups[0]['lr'])

            gm = {'gen_exact_match': float('nan'), 'gen_exec_match': float('nan'),
                  'gen_abstention_precision': float('nan'), 'gen_abstention_recall': float('nan')}
            if (epoch + 1) % self.cfg['eval_gen_every_n_epochs'] == 0:
                gm = evaluate_generation(self.model, self.tokenizer, GEN_EVAL_SAMPLE,
                                          self.schema_ddl, self.cfg, self.duckdb_con)
            for k in ('gen_exact_match', 'gen_exec_match', 'gen_abstention_precision', 'gen_abstention_recall'):
                self.history[k].append(gm[k])

            is_best = vm['loss'] < self.best_val_loss
            if is_best:
                self.best_val_loss = vm['loss']; self.best_epoch = epoch + 1
            self._save(epoch + 1, is_best)

            self._log(f'[seed{self.seed}] ep {epoch+1}/{self.cfg["max_epochs"]} | tr {tr_loss:.4f} '
                      f'| val {vm["loss"]:.4f} (best {self.best_val_loss:.4f}@{self.best_epoch}) '
                      f'| ppl {vm["perplexity"]:.2f} | tok-acc {vm["token_acc"]:.4f} '
                      f'| exact {gm["gen_exact_match"]:.3f} '
                      f'| exec {gm["gen_exec_match"]:.3f} | abst P/R {gm["gen_abstention_precision"]:.2f}/'
                      f'{gm["gen_abstention_recall"]:.2f} | {time.time()-t0:.0f}s')

            # Early stopping after epoch 2: max_epochs=3, but only attempt epoch 3 if epoch 2
            # improved meaningfully on epoch 1 -- otherwise the 3rd epoch is very unlikely to be
            # worth its compute. Only fires right after epoch 2 specifically (not every epoch),
            # and only when there's a 3rd epoch left to skip -- on resume, history already has
            # both epochs' val_loss loaded, so this check reproduces the same decision rather
            # than needing separate resume-state tracking.
            if (epoch + 1) == 2 and self.cfg['max_epochs'] > 2 and len(self.history['val_loss']) >= 2:
                val_loss_ep1, val_loss_ep2 = self.history['val_loss'][0], self.history['val_loss'][1]
                rel_improvement = (val_loss_ep1 - val_loss_ep2) / val_loss_ep1
                threshold = self.cfg['sft_early_stop_min_relative_improvement']
                if rel_improvement < threshold:
                    self._log(f'[seed{self.seed}] early stopping after epoch 2 -- val_loss improved only '
                              f'{rel_improvement:.1%} over epoch 1 (threshold {threshold:.1%}), skipping epoch 3')
                    break
                else:
                    self._log(f'[seed{self.seed}] epoch 2 improved {rel_improvement:.1%} over epoch 1 '
                              f'(>= {threshold:.1%} threshold) -- continuing to epoch 3')
        self._tee.close()
        return self.best_val_loss, self.best_epoch

## Multi-seed driver

Runs each seed in `CFG['seeds']` into its own isolated `sft_outputs/seed_<seed>/` directory,
skipping any seed whose `history.json` already shows `max_epochs` completed epochs -- safe to
re-run this cell after any interruption.

In [ ]:
def seed_out_dir(seed):
    return os.path.join(OUT_BASE, f'seed_{seed}')


def seed_is_done(seed):
    hp = os.path.join(seed_out_dir(seed), 'history.json')
    if not os.path.exists(hp):
        return False
    h = json.load(open(hp))
    return len(h['train_loss']) >= CFG['max_epochs']


duckdb_con = None
if CFG['run_execution_eval']:
    import duckdb
    local_duckdb = _copy_local(DRIVE_DUCKDB, 'train.duckdb') if ON_COLAB else DRIVE_DUCKDB
    duckdb_con = duckdb.connect(local_duckdb, read_only=True)
    print('execution-eval DB connected:', local_duckdb)

BASELINE = json.load(open(_baseline_path())) if os.path.exists(_baseline_path()) else None
TEST_BASELINE = json.load(open(_test_baseline_path())) if os.path.exists(_test_baseline_path()) else None

GRID_T0 = time.time()
for seed in CFG['seeds']:
    seed_test_path = os.path.join(seed_out_dir(seed), 'test_eval.json')
    training_done = seed_is_done(seed)
    if training_done and os.path.exists(seed_test_path):
        print(f'[skip] seed {seed} already complete (training + test eval)'); continue

    print(f'\n================  seed {seed}  ================')
    t0 = time.time()
    trainer = SFTTrainer(CFG, seed, train_rows, dev_rows, test_rows, schema_ddl, seed_out_dir(seed), duckdb_con)
    if BASELINE is None:
        BASELINE = get_or_compute_baseline(trainer.model, trainer.tokenizer, trainer.dev_loader,
                                            GEN_EVAL_SAMPLE, schema_ddl, CFG, duckdb_con)

    if not training_done:
        best_loss, best_epoch = trainer.train()
        print(f'---- seed {seed} done in {(time.time()-t0)/60:.1f} min | best val_loss {best_loss:.4f} @ epoch {best_epoch} ----')
    else:
        print(f'[skip] seed {seed} training already complete -- backfilling test eval only')

    if TEST_BASELINE is None:
        TEST_BASELINE = get_or_compute_test_baseline(trainer.model, trainer.tokenizer, trainer.test_loader,
                                                       TEST_GEN_SAMPLE, schema_ddl, CFG, duckdb_con)
    if not os.path.exists(seed_test_path):
        best_dir = os.path.join(seed_out_dir(seed), 'best')
        test_result = evaluate_checkpoint_on_test(trainer.model, best_dir, 'test_eval_ckpt', trainer.tokenizer,
                                                    trainer.test_loader, TEST_GEN_SAMPLE, schema_ddl, CFG, duckdb_con)
        json.dump(test_result, open(seed_test_path, 'w'), indent=2)
        print(f'[test] seed {seed} best-checkpoint test metrics -> {seed_test_path}')
        print(json.dumps(test_result, indent=2))

    del trainer
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()
print(f'\n=====  all seeds complete in {(time.time()-GRID_T0)/3600:.2f} h  =====')
if BASELINE is None or TEST_BASELINE is None:
    print('[baseline] WARNING: dev and/or test baseline still not computed (every seed was already '
          'complete when this cell ran) -- call get_or_compute_baseline(...) / '
          'get_or_compute_test_baseline(...) manually with a live trainer.model to backfill.')

## Progress (safe to run anytime, even mid-training)

In [ ]:
if os.path.exists(_baseline_path()):
    _b = json.load(open(_baseline_path()))
    print(f"{'baseline':<6} {'(no adapter)':<12} {_b['val_loss']:<14.4f} {_b['val_perplexity']:<10.2f} {'--':<10}")
print(f"{'seed':<6} {'epochs_done':<12} {'best_val_loss':<14} {'best_ppl':<10} {'best_epoch':<10}")
for seed in CFG['seeds']:
    hp = os.path.join(seed_out_dir(seed), 'history.json')
    if not os.path.exists(hp):
        print(f'{seed:<6} not started'); continue
    h = json.load(open(hp))
    n = len(h['train_loss'])
    epochs_str = f"{n}/{CFG['max_epochs']}"
    best_idx = int(np.argmin(h['val_loss'])) if h['val_loss'] else None
    best = h['val_loss'][best_idx] if best_idx is not None else float('nan')
    best_ppl = h['val_perplexity'][best_idx] if best_idx is not None else float('nan')
    best_ep = (best_idx + 1) if best_idx is not None else 0
    print(f'{seed:<6} {epochs_str:<12} {best:<14.4f} {best_ppl:<10.2f} {best_ep:<10}')

## Cross-seed summary, dev split (training-time diagnostic)

Reports best-checkpoint **dev** metrics per seed and their mean/std -- useful as a live signal
while training is still running, but `dev` was used for checkpoint selection, so treat this as a
diagnostic, not the final number. See "Cross-seed summary, test split" below for the untouched
comparison.

In [ ]:
baseline = json.load(open(_baseline_path())) if os.path.exists(_baseline_path()) else None

finals = []
for seed in CFG['seeds']:
    hp = os.path.join(seed_out_dir(seed), 'history.json')
    if not os.path.exists(hp):
        continue
    h = json.load(open(hp))
    if len(h['train_loss']) < CFG['max_epochs']:
        continue
    bi = int(np.argmin(h['val_loss']))
    finals.append(dict(seed=seed, val_loss=h['val_loss'][bi], val_perplexity=h['val_perplexity'][bi],
                        token_acc=h['val_token_acc'][bi],
                        exact_match=h['gen_exact_match'][bi], exec_match=h['gen_exec_match'][bi],
                        abst_p=h['gen_abstention_precision'][bi], abst_r=h['gen_abstention_recall'][bi]))

df = pd.DataFrame(finals)
print(df.to_string(index=False) if len(df) else 'no completed seeds yet')

if baseline is not None:
    print('\nfrozen base model (no adapter):')
    print(f"  val_loss={baseline['val_loss']:.4f} val_perplexity={baseline['val_perplexity']:.2f} "
          f"token_acc={baseline['val_token_acc']:.4f} "
          f"exact_match={baseline['gen_exact_match']:.3f} exec_match={baseline['gen_exec_match']:.3f} "
          f"abst_p={baseline['gen_abstention_precision']:.2f} abst_r={baseline['gen_abstention_recall']:.2f}")

if len(df) > 1:
    print('\nmean +/- std across seeds:')
    for c in df.columns:
        if c == 'seed':
            continue
        print(f'  {c}: {df[c].mean():.4f} +/- {df[c].std():.4f}')

if baseline is not None and len(df):
    print('\nSFT impact (best-checkpoint mean vs frozen baseline; positive = SFT better):')
    print(f"  val_loss:    {baseline['val_loss'] - df['val_loss'].mean():+.4f}  (loss reduction, so positive = improvement)")
    print(f"  val_perplexity: {baseline['val_perplexity'] - df['val_perplexity'].mean():+.2f}  (perplexity reduction, so positive = improvement)")
    print(f"  token_acc:   {df['token_acc'].mean() - baseline['val_token_acc']:+.4f}")
    print(f"  exact_match: {df['exact_match'].mean() - baseline['gen_exact_match']:+.4f}")
    print(f"  exec_match:  {df['exec_match'].mean() - baseline['gen_exec_match']:+.4f}")
    print(f"  abst_p:      {df['abst_p'].mean() - baseline['gen_abstention_precision']:+.4f}")
    print(f"  abst_r:      {df['abst_r'].mean() - baseline['gen_abstention_recall']:+.4f}")

## Cross-seed summary, test split (the number to trust)

Same comparison as above, but computed on the `test` split that was never touched during training
or checkpoint selection. Requires the "Held-out test split" section's per-seed evaluation to have
run for at least one seed.

In [ ]:
test_baseline = json.load(open(_test_baseline_path())) if os.path.exists(_test_baseline_path()) else None

test_finals = []
for seed in CFG['seeds']:
    tp = os.path.join(seed_out_dir(seed), 'test_eval.json')
    if not os.path.exists(tp):
        continue
    t = json.load(open(tp))
    test_finals.append(dict(seed=seed, **t))

tdf = pd.DataFrame(test_finals)
print(tdf.to_string(index=False) if len(tdf) else 'no seeds with a completed test evaluation yet')

if test_baseline is not None:
    print('\nfrozen base model (no adapter), test split:')
    print(f"  val_loss={test_baseline['val_loss']:.4f} val_perplexity={test_baseline['val_perplexity']:.2f} "
          f"token_acc={test_baseline['val_token_acc']:.4f} "
          f"exact_match={test_baseline['gen_exact_match']:.3f} exec_match={test_baseline['gen_exec_match']:.3f} "
          f"abst_p={test_baseline['gen_abstention_precision']:.2f} abst_r={test_baseline['gen_abstention_recall']:.2f}")

if len(tdf) > 1:
    print('\nmean +/- std across seeds (test split):')
    for c in tdf.columns:
        if c == 'seed':
            continue
        print(f'  {c}: {tdf[c].mean():.4f} +/- {tdf[c].std():.4f}')

if test_baseline is not None and len(tdf):
    print('\nSFT impact on the held-out TEST split (best-checkpoint mean vs frozen baseline; positive = SFT better):')
    print(f"  val_loss:    {test_baseline['val_loss'] - tdf['val_loss'].mean():+.4f}  (loss reduction, so positive = improvement)")
    print(f"  val_perplexity: {test_baseline['val_perplexity'] - tdf['val_perplexity'].mean():+.2f}  (perplexity reduction, so positive = improvement)")
    print(f"  token_acc:   {tdf['val_token_acc'].mean() - test_baseline['val_token_acc']:+.4f}")
    print(f"  exact_match: {tdf['gen_exact_match'].mean() - test_baseline['gen_exact_match']:+.4f}")
    print(f"  exec_match:  {tdf['gen_exec_match'].mean() - test_baseline['gen_exec_match']:+.4f}")
    print(f"  abst_p:      {tdf['gen_abstention_precision'].mean() - test_baseline['gen_abstention_precision']:+.4f}")
    print(f"  abst_r:      {tdf['gen_abstention_recall'].mean() - test_baseline['gen_abstention_recall']:+.4f}")

## Plot: dev metrics vs frozen baseline, per seed, per epoch

Visual version of the **dev-split** (training-time) impact analysis above -- each seed's dev-set
curve across training epochs, with the frozen dev baseline as a horizontal reference line. Saved
to `sft_outputs/sft_vs_baseline.png`. For the trustworthy final number, see the test-split summary
above instead.

In [ ]:
import matplotlib.pyplot as plt

baseline = json.load(open(_baseline_path())) if os.path.exists(_baseline_path()) else None
metrics_to_plot = [('val_loss', 'dev loss (lower better)'), ('val_token_acc', 'dev token accuracy'),
                    ('gen_exact_match', 'exact SQL match'), ('gen_exec_match', 'execution match')]

fig, axes = plt.subplots(2, 2, figsize=(11, 8))
for ax, (key, title) in zip(axes.flat, metrics_to_plot):
    for seed in CFG['seeds']:
        hp = os.path.join(seed_out_dir(seed), 'history.json')
        if not os.path.exists(hp):
            continue
        h = json.load(open(hp))
        if not h[key]:
            continue
        epochs = list(range(1, len(h[key]) + 1))
        ax.plot(epochs, h[key], marker='o', label=f'seed {seed}')
    if baseline is not None and key in baseline:
        ax.axhline(baseline[key], linestyle='--', color='black', label='frozen baseline')
    ax.set_title(title); ax.set_xlabel('epoch'); ax.legend(fontsize=8)
fig.tight_layout()
fig_path = os.path.join(OUT_BASE, 'sft_vs_baseline.png')
fig.savefig(fig_path, dpi=110)
print('saved ->', fig_path)
plt.show()